**FYI – Reminder to Dom: This lab was hard. Lots of the lab asked for actions that required other steps to be done that weren't really discussed, e.g. dim tables that I never created as an example. We touched on lots of it in the lecture as well, but the lab requires us to go into detail on things we only touched on, without really making it clear, such as the notes I made in Q2. Also see Q13, Q14 and Q17. Because of that, it's worth noting that the concepts below are ones I know are also needed and were touched on, but aren't in the lab. I don't think I make reference in any of the q- # notes below, that I constantly need to run the tables etc. in the terminal after each change.**

**Not used in this lab (Week 4, Day 1: dbt Fundamentals)**

- **ephemeral:** only appears as an option in Q11. No model is built as ephemeral.
- **`tests/` folder (singular SQL tests):** not used. All tests in the lab are generic YAML tests (`not_null`, `unique`, `accepted_values`) in `schema.yml`. Anyelle's `assert_no_negative_revenue.sql` is an extra example.
- **`seeds/` (`dbt seed`):** not used. The raw data was loaded into `dev.duckdb` by hand with `CREATE TABLE ... read_csv_auto` in the DuckDB UI.
- **`snapshots/`, `macros/`, `analyses/`:** created by `dbt init`, but not used.
- **`dbt docs generate`:** not used. Q15 writes descriptions in YAML, but the docs website is never built.
- **`dbt build`:** not used. The lab uses `dbt run` and `dbt test` separately.
- **Packages (e.g. `dbt_utils`):** not used.
- **Tests and docs on sources:** optional, not asked for.
- **`models/example/`:** dbt's starter models. Not part of BritMart, safe to delete.

## Running Ironhack notebooks in VS Code

### One-off setup (already done)

Your `.venv_airflow` environment already has `ipykernel` and `pandas` installed, so you don't need to repeat this. It's here in case you ever make a new environment or need a new package.

1. In the VS Code terminal, make sure you're in `~/data_course`.
2. Switch the environment on: `source .venv_airflow/bin/activate` (the prompt should start with `(.venv_airflow)`).
3. Install what notebooks need: `pip install ipykernel pandas`
4. For any other package a lab needs later (for example numpy), use the same pattern: `pip install <package name>`

### Every time you add a new notebook

Each new .ipynb file needs its kernel picking once. After that, VS Code remembers it for that file.

1. **Drag it in.** Drag the downloaded .ipynb from Windows into your `01_notebooks` folder in the VS Code Explorer (left-hand panel). Rename it if you like (right-click → Rename), for example replacing DATE with the date.
2. **Open it.** Click the file in the Explorer.
3. **Click Select Kernel.** It's in the top-right corner of the notebook. (If it already says `.venv_airflow`, skip to step 7.)
4. **Click Python Environments…**
5. **Pick .venv_airflow.** Choose the line ending `.venv_airflow/bin/python`. Ignore the one marked "Recommended" (`/usr/bin/python3`): that's Ubuntu's own Python and it won't work.
6. **Check the top-right.** It should now say `.venv_airflow`.
7. **Run the import cell.** Click ▷ next to the import cell. A green tick and [1] mean it's working.

## Part 1 — dbt Architecture and Core Concepts

### Q. 1 — Identify source() and ref()

BritMart has a raw table created outside dbt:

```python
raw.orders
```

and a dbt model named:

```python
stg_orders
```

Complete these two expressions:

```sql
{{ ______('raw', 'orders') }}
{{ ______('stg_orders') }}
```

Then explain when each one is used.

========================================================================
ANSWER Q1
========================================================================

{{source('raw', 'orders')}}

====================================

{{ref('stg_orders')}}

====================================

When each one is used:

The first table is from a raw source file, so the script needs to read from source. The second model is either a view or a table that has been created inside of dbt, so this new query needs to refer to the model that exists within the dbt script.

'source() is used for raw tables that were loaded into the database outside of dbt, like raw.orders. They must be declared in a sources YAML file first.
ref() is used to point at another dbt model in the project, like stg_orders, whether it was built as a view or a table. It also tells dbt the models depend on each other, so dbt runs them in the right order and can show the lineage.'

========================================================================

### Q. 2 — Create a Source Definition

BritMart has this source table:

```python
Schema: raw
Table: orders
```

Create a complete YAML source definition named:

```python
britmart_raw
```

In [ ]:
version: 2
# Since dbt version 1.5, version: 2 at the top of YAML files like sources.yml and schema.yml is optional, so dbt will read the file fine without it. 
# Older versions of dbt required it. 

sources:
  - name: britmart_raw
    description: Raw britmart orders, exactly as they arrived.
    schema: raw
    tables:
      - name: orders

# The above works under the assumption that dbt is reading from a database. dbt is always connected to a database through `profiles.yml` 
# (mine is `dev.duckdb`), but Ironhack's answer only works if the raw data is already a table in that database, in a schema called `raw`. 
# That's the normal real-world setup: a loading tool like Airflow, Fivetran or a Python script copies the data into the warehouse (e.g. `raw.orders`), 
# and dbt doesn't load anything itself. It just reads that table with `source()` and transforms it.
# In the example below I'm connecting to something 'external' to the database, for example purposes. 

# sources:
#   - name: britmart_raw
#     description: Raw britmart orders, exactly as they arrived.
#     meta:
#       external_location: "real_file_path"
#     tables:
#       - name: orders

# ======================================================================
# dbt notes: missing raw schema
# ======================================================================
#
# 1. For Q2, do I need to build source data like Anyelle did?
#    No. Q2 only asks for the YAML (sources.yml with schema: raw and
#    table orders), which dbt can read without any data.
#    But Ironhack's answer assumes the raw schema and orders table
#    ALREADY EXIST in the database, loaded there by a tool like Airflow
#    or Fivetran, because dbt doesn't load data itself.
#    In my project there is NO raw schema in dev.duckdb. It only has
#    main, with dbt's two example models. So any model that reads the
#    source (from Q3) will fail with "table not found" until the data
#    is there.
#
# 2. How do I open dev.duckdb in the DuckDB UI?
#    1) Install once:
#         curl https://install.duckdb.org | sh
#    2) Add it to PATH:
#         echo 'export PATH="/home/dominicbrookes/.duckdb/cli/latest:$PATH"' >> ~/.bashrc
#    3) Open a NEW terminal, activate the venv, then check:
#         source ~/data_course/.venv_airflow/bin/activate
#         duckdb --version
#    4) Go to the project folder and start the UI:
#         cd ~/data_course/26.11.14_britmart_dbt
#         duckdb -ui dev.duckdb
#       It opens at localhost:4213.
#    5) Press Ctrl+C to close it before running dbt, or you'll get a
#       lock error.
#
# 3. Is it separate for every new project folder?
#    Yes. Each dbt project has its own dev.duckdb, and the UI opens
#    whichever one you give it. For a new project, cd into its folder
#    and run: duckdb -ui dev.duckdb
#    The UI's notebooks are shared across all projects, and only one UI
#    can run at a time.
#
# 4. What are my options for Q3?
#    1) Write it, don't run it.
#       Create stg_orders.sql and copy it into the notebook.
#       Nothing to set up, but you can't test it.
#    2) Load the data into dev.duckdb (recommended).
#       Copy the CSVs into raw_data, then run this in the UI:
#         CREATE SCHEMA IF NOT EXISTS raw;
#         CREATE OR REPLACE TABLE raw.orders AS
#         SELECT * FROM read_csv_auto('raw_data/orders.csv');
#       sources.yml stays as it is (schema: raw) and dbt run works.
#       This matches Ironhack's setup.
#    3) Point the source at the CSV.
#       Use meta: external_location: "raw_data/{name}.csv" in
#       sources.yml. DuckDB reads the file directly, so there's nothing
#       to load (Anyelle's approach).
#    Whichever you pick, check the data has the columns Q3 needs
#    (order_id, customer_id, amount_gbp, status) with:
#         DESCRIBE raw.orders;
# ======================================================================

# ======================================================================
# How I added the raw schema to dev.duckdb
# ======================================================================
#
# Why: sources.yml uses schema: raw, but dev.duckdb had no raw schema,
# so any model reading the source would fail with "table not found".
#
# 1. Copied Anyelle's CSVs into my project's raw_data folder:
#      cp ~/anyellequeiroz_w08d03/raw_data/*.csv ~/data_course/26.11.14_britmart_dbt/raw_data/
#
# 2. Started the DuckDB UI from the project folder (the CSV paths are
#    read from where the UI is started):
#      cd ~/data_course/26.11.14_britmart_dbt
#      duckdb -ui dev.duckdb
#    "xdg-open: not found" just means WSL can't open the Windows browser
#    itself, so I opened localhost:4213 in my browser manually.
#    Leave the terminal running (dev D prompt) while using the UI.
#
# 3. In the UI: Notebooks -> + (new notebook), pasted this into a cell
#    and ran it with Ctrl+Enter:
#      CREATE SCHEMA IF NOT EXISTS raw;
#
#      CREATE OR REPLACE TABLE raw.orders AS
#      SELECT * FROM read_csv_auto('raw_data/orders.csv');
#
#      CREATE OR REPLACE TABLE raw.customers AS
#      SELECT * FROM read_csv_auto('raw_data/customers.csv');
#
# 4. Checked it worked: dev -> raw -> orders and customers now show on
#    the left. Then ran:
#      DESCRIBE raw.orders;
#      SELECT * FROM raw.orders LIMIT 10;
#
# 5. Closed the UI before running dbt (.quit or Ctrl+C in the terminal),
#    otherwise dbt gets a lock error on dev.duckdb.
#
# Notes:
# - dbt builds its models into the main schema, so it never touches raw.
# - dbt won't refresh raw. If a CSV changes, rerun the CREATE OR REPLACE
#   lines.
# - orders.csv has a row with no order_id on purpose, for Q3's
#   "keep only rows where order_id is not null".
# ======================================================================


# ======================================================================
# Things called the same but taht are different (schema, tests)
# ======================================================================


# FYI: schema.yml is not to be confused with creating a schema in dbt. Schema.yml is improtant documentation that is useful for test and docuemtnation. Really each 
# of the model subfolders should have one of these files.
# The tests that are in schema.yml should also not be confused with the tests that are in the subfolder called 'tests'. '**Generic tests** (`not_null`, `unique`) are 
# built-in templates you just name in `schema.yml`. **Singular tests** are your own SQL queries, so they need their own `.sql` file in `tests/`, because YAML can't hold SQL. Either way, a test passes if its query returns no rows.'
# ======================================================================

### Q. 3 — Create a Simple dbt Model

Create:

```python
models/staging/stg_orders.sql
```

Requirements:

- read from `britmart_raw.orders`;
- select `order_id`, `customer_id`, `amount_gbp`, and `status`;
- keep only rows where `order_id` is not null.

In [ ]:
select
    order_id,
    customer_id,
    amount_gbp,
    status
from {{source('britmart_raw', 'orders')}}
where order_id is not null

# ======================================================================
# What happens when I run: dbt run --select stg_orders
# (run from ~/data_course/26.11.14_britmart_dbt)
# ======================================================================
# 1. dbt finds the model. --select stg_orders means only build this one.
# 2. dbt replaces the Jinja. It looks up britmart_raw -> orders in
#    sources.yml, sees schema: raw, and turns the source() line into a
#    real table name, roughly "dev"."raw"."orders".
#    The finished SQL is saved in target/compiled/.../stg_orders.sql
# 3. No materialisation is set, so dbt uses the default: a VIEW.
#    It wraps my select in: create view main.stg_orders as ...
# 4. dbt runs it on dev.duckdb. The view is created in the main schema,
#    next to dbt's two example models.
# 5. dbt reports back, e.g.:
#    1 of 1 OK created sql view model main.stg_orders
#
# Check: open the DuckDB UI -> dev -> main -> stg_orders.
# The row with no order_id should be gone.
#
# If it fails:
# - "raw.orders does not exist"     -> the load into raw didn't happen
# - "column amount_gbp not found"   -> the CSV uses a different column
#                                      name (check: DESCRIBE raw.orders;)
# - lock error                      -> DuckDB or the UI still has
#                                      dev.duckdb open
# ======================================================================

### Q. 4 — MCQ: ref()

What is the main purpose of `ref()` in dbt?

A. Connect directly to any external database without configuration

B. Replace all dbt tests

C. Refer to another dbt model and create a model dependency

D. Create an Airflow DAG

========================================================================
ANSWER Q4
========================================================================

C. Refer to another dbt model and create a model dependency.

========================================================================

## Part 2 — Project Structure and Staging Models

### Q. 5 — Organise a Small dbt Project

Create a sensible folder structure for:

```python
sources
staging models
intermediate models
mart models
tests/documentation YAML
```

Use a simple text tree.

========================================================================
ANSWER Q5
========================================================================

britmart_dbt/

├── dbt_project.yml

└── models/

    ├── staging/

    │   ├── stg_orders.sql

    │   └── schema.yml

    ├── intermediate/

    │   └── int_completed_orders.sql

    └── marts/

        ├── fct_daily_sales.sql

        └── schema.yml

└── sources/

    ├── sources.yml

└── tests/

    ├── documentation YAML
========================================================================

### Q. 6 — Build a Staging Model

Create an independent staging model from this source:

```python
britmart_raw.customers
```

Required output columns:

```python
customer_id
customer_name
city
```

Rules:

- trim spaces from `customer_name`;
- convert `city` to upper case.

In [ ]:
# After adding the customers table to sources.yml

select
    trim(customer_name) as customer_name,
    customer_id,
    upper(city) as city
from {{source('britmart_raw', 'customers')}}

### Q. 7 — Build a Staging Model with Type Conversion

Source columns:

```python
order_id
order_date
amount_text
```

Create a model that returns:

```python
order_id
order_date
amount_gbp
```

Convert `amount_text` to a decimal value.

In [ ]:
-- try_cast instead of cast. If any amount_text value isn’t a valid number (like "abc" or "£4.25"), cast makes the whole model fail, while try_cast puts null for that row and carries on. Either is fine for the lab; cast is what the question expects. 
select
order_id,
order_date,
try_cast(amount_gbp as decimal(10,2)) as amount_gbp
from {{source('britmart_raw','orders')}}

### Q. 8 — MCQ: Staging Model

What is a good responsibility for a staging model?

A. Standardise source columns and basic types

B. Create Airflow retry rules

C. Build the final executive dashboard

D. Replace the source system

========================================================================
ANSWER Q8
========================================================================

A. Standardise source columns and basic types.

========================================================================

## Part 3 — Intermediate Models, Marts, and Materialisations

### Q. 9 — Build an Intermediate Model

Assume these dbt models already exist:

```python
stg_orders
stg_customers
```

Create an intermediate model that joins them by `customer_id`.

Return:

```python
order_id
customer_id
customer_name
city
amount_gbp
status
```

In [ ]:
select
stg_orders.order_id,
stg_orders.customer_id,
stg_customers.customer_name,
stg_customers.city,
stg_orders.amount_gbp,
stg_orders.status
from {{ref('stg_orders')}}
left join {{ref('stg_customers')}} on stg_orders.customer_id = stg_customers.customer_id

### Q. 10 — Build a Mart Model

Create a mart model named:

```python
fct_daily_sales
```

Input model:

```python
int_orders_enriched
```

Requirements:

- keep only `status = 'complete'`;
- group by `order_date` and `city`;
- calculate `order_count`;
- calculate `revenue_gbp`.

In [ ]:
select
q9.order_date,
q9.city,
count(q9.order_id) as order_count,
sum(q9.amount_gbp) as revenue_gbp
from {{ref('q9')}}
where q9.status = 'complete'
group by q9.order_date, q9.city

-- The only time this breaks is if you rename the file. In which case you need...
-- select
-- order_date,
-- city,
-- count(order_id) as order_count,
-- sum(amount_gbp) as revenue_gbp
-- from {{ref('q9')}}
-- where status = 'complete'
-- group by order_date, city

### Q. 11 — Choose a Materialisation

Choose the better starting materialisation for each case:

```python
A. Small reusable model that is cheap to calculate each time.
B. Large final mart queried often.
C. Model that should store only new/changed data after the first run.
D. Small helper logic that does not need its own stored table.
```

Choose from:

```python
view
table
incremental
ephemeral
```

========================================================================
ANSWER Q11
========================================================================

A. Small reusable model that is cheap to calculate each time.

view.

====================================

B. Large final mart queried often.

table.

====================================

C. Model that should store only new/changed data after the first run.

incremental.

====================================

D. Small helper logic that does not need its own stored table.

ephemeral.

========================================================================

### Q. 12 — MCQ: Mart Model

Which model is the strongest example of a mart?

A. A raw source declaration

B. A temporary source file

C. A dbt profile password

D. A table of final daily sales metrics for reporting

========================================================================
ANSWER Q12
========================================================================

D. A table of final daily sales metrics for reporting.

========================================================================

## Part 4 — Incremental Models, Tests, Documentation, and Lineage

### Q. 13 — Write an Incremental Model

Create a model with:

```python
materialized = incremental
unique_key = order_id
```

Input:

```python
stg_orders
```

Use `updated_at` so an incremental run reads only rows newer than the maximum `updated_at` already stored.

In [ ]:
# Incremental models aren't always better than rebuilding from scratch.
# They're more complicated (config, is_incremental(), {{ this }} and a
# reliable updated_at column), and changing the logic only affects new rows
# until you run dbt run --full-refresh. They don't notice rows deleted in
# the source, and they can miss late-arriving rows that have an older
# updated_at. For small data they save no real time. The usual approach is
# to start with a view, switch to a table when queries get slow, and only
# use incremental when rebuilding the table itself gets slow, typically
# millions of rows that mostly just keep growing, like orders or events.

# Note: for Q13 I had to go back to stg_orders.sql and add updated_at to
# the select, because the incremental model filters on updated_at and
# stg_orders didn't include it (the same happened with order_date for Q10).
# Adding extra columns to a staging

{{ config(
    materialized='incremental',
    unique_key='order_id'
) }}

select
    order_id,
    customer_id,
    order_date,
    amount_gbp,
    status,
    updated_at
from {{ ref('stg_orders') }}

{% if is_incremental() %}
where updated_at > (select max(updated_at) from {{ this }})
{% endif %}

### Q. 14 — Add Data Tests

Create YAML tests for this model:

```python
dim_customer
```

Rules:

```python
customer_id -> not_null and unique
customer_name -> not_null
city -> accepted values: LONDON, MANCHESTER, BRISTOL, LEEDS
```

In [ ]:
# I created dim_customer so Q14's tests have a real model to check
# (without it, dbt test has nothing to run against).
# It reads straight from stg_customers: a mart can ref a staging model
# directly, and an intermediate model is only needed when there's work
# to do in between, like a join.

version: 2

models:
  - name: fct_daily_sales
    description: Daily sales by city, completed orders only.
    columns:
      - name: order_date
        description: The day the orders were placed.
        data_tests:
          - not_null
      - name: city
        description: Customer city in upper case.
      - name: order_count
        description: Number of completed orders for that day and city.
        data_tests:
          - not_null
      - name: revenue_gbp
        description: Total value of completed orders for that day and city, in pounds.

  - name: dim_customer
    description: One row per customer, built straight from stg_customers.
    columns:
      - name: customer_id
        description: Unique ID for each customer.
        data_tests:
          - not_null
          - unique
      - name: customer_name
        description: Customer name with extra spaces removed.
        data_tests:
          - not_null
      - name: city
        description: Customer city in upper case.
        data_tests:
          - accepted_values:
              arguments:
                values: ['LONDON', 'MANCHESTER', 'BRISTOL', 'LEEDS']

  - name: fct_orders_incremental
    description: Incremental orders table. After the first run, only rows with a newer updated_at are processed.
    columns:
      - name: order_id
        description: Unique ID for each order (the unique_key).
        data_tests:
          - not_null
          - unique
      - name: customer_id
        description: ID of the customer who placed the order.
      - name: order_date
        description: Date the order was placed.
      - name: amount_gbp
        description: Order value in pounds.
      - name: status
        description: Order status, e.g. complete.
      - name: updated_at
        description: When the order was last changed. Used to pick up new rows on incremental runs.

### Q. 15 — Document a Model

Create YAML documentation for:

```python
fct_daily_sales
```

Include:

```python
model description
order_date description
city description
revenue_gbp description
```

In [ ]:
# q14 contains the answer to q15.

### Q. 16 — MCQ: Data Test

When does a dbt data test pass?

A. When every model is materialised as a table

B. When its test query returns no failing rows

C. When Airflow is running

D. When a source has no columns

========================================================================
ANSWER Q16
========================================================================

B. When its test query returns no failing rows.

 FYI:
 dbt test checks your data against the rules you've written. It collects
 every test in the project (generic tests in schema.yml and sources.yml,
 plus any SQL files in the tests/ folder) and turns each one into a SQL
 query that looks for bad rows. For example, unique on
 dim_customer.customer_id becomes roughly: select customer_id from
 main.dim_customer group by customer_id having count(*) > 1. It runs each
 query on the models already built in dev.duckdb: 0 rows found means PASS
 and any rows found means FAIL (e.g. FAIL 2 means two rows broke the rule).
 dbt test doesn't build or change anything, it only reads, which is why
 you need dbt run first.

========================================================================

## Optional Practice

### Q. 17 — Build an Independent Three-Model Flow

Create three complete model files:

```python
stg_orders.sql
int_completed_orders.sql
fct_city_sales.sql
```

Use this source:

```python
britmart_raw.orders
```

Required logic:

```python
stg_orders:
select and standardise order fields

int_completed_orders:
keep only completed orders

fct_city_sales:
group by city and calculate order count and revenue
```

In [ ]:
# stg_orders.sql already exists from an earlier question. However, it does not contain a column called city, and adding one would mean
# adding city to the source CSV and reloading raw.orders in the DuckDB UI. So instead I'll join the stg_customers.sql I have created
# (in int_completed_orders, so staging stays one source per model). I'll also add in some trims etc. to match the Ironhack answer.
# Technically these could have been, and maybe should have been, done in my original staging tables anyway (the bronze layer (staging in dbt) either
# has more standardisation in dbt, or every bronze layer should really work the same).

# I created int_completed_orders.sql and updated schema.yml (I do not need to re-run the schema.yml each time.)
select
stg_orders.order_id,
stg_orders.customer_id,
cast(stg_orders.amount_gbp as decimal (10,2)) as amount_gbp,
stg_orders.order_date,
lower(stg_orders.status) as status,
trim(stg_customers.city) as city
from {{ref('stg_orders')}}
left join {{ref('stg_customers')}} on stg_orders.customer_id = stg_customers.customer_id
where lower(stg_orders.status) = 'complete'

# I created fct_city_sales.sql and updated schema.yml (I do not need to re-run the schema.yml each time.)
select
int_completed_orders.city,
count(int_completed_orders.order_id) as order_count,
sum(int_completed_orders.amount_gbp) as revenue_gbp
from {{ref('int_completed_orders')}}
group by int_completed_orders.city

### Q. 18 — Find a Lineage Problem

A model contains:

```sql
select *
from analytics.stg_orders
```

instead of:

```sql
select *
from {{ ref('stg_orders') }}
```

Explain what dbt loses when the hard-coded table name is used.

========================================================================
ANSWER Q18
========================================================================

Hard-coding analytics.stg_orders means dbt doesn't know this model depends on stg_orders.
It loses the run order (stg_orders may not be built first), the lineage graph, and the
dev/prod switching that ref() handles. If stg_orders is renamed, dbt won't catch it.

Caution: I prefix columns with the table name (stg_orders.order_id) instead of an alias.
That's fine, but the ref() must stay in the from/join lines. And if a model is renamed,
every prefix needs updating.

========================================================================

### Q. 19 — Design Tests for a Sales Mart

Model columns:

```python
order_date
city
order_count
revenue_gbp
```

Suggest at least four useful tests using only dbt testing concepts covered today.

========================================================================
ANSWER Q19
========================================================================

1. not_null on order_date: every row needs a date.
2. not_null on city: every row needs a city.
3. accepted_values on city: LONDON, MANCHESTER, BRISTOL, LEEDS.
4. not_null on order_count: every day/city row must have a count.
5. not_null on revenue_gbp: every day/city row must have revenue.

Extra: a singular test in tests/ that selects rows where revenue_gbp < 0 or
order_count <= 0. It passes if the query returns no rows.

========================================================================

### Q. 20 — Choose What to Run

A project contains:

```python
models/staging/
models/intermediate/
models/marts/
```

Write dbt commands to:

- run all staging models;
- test all mart models;
- run one model named `fct_daily_sales`.

In [ ]:
dbt run --select staging
dbt run --select intermediate
# Intermediate models don't need running first if they've already been run and haven't changed since.
dbt test --select marts
dbt run --select fct_daily_sales